# AudioRestore: Fine-tuning SonicMaster for Codec Audio Restoration

**CS 614 Final Project** — Fine-tune [SonicMaster](https://github.com/AMAAI-Lab/SonicMaster) (ICLR 2026, 0.9B params) on codec-degraded audio.

**Requirements**: Colab with A100 GPU (Colab Pro) or T4 (free tier with fp16).

## Workflow
1. Install dependencies
2. Mount Google Drive for persistent storage
3. Download pretrained SonicMaster checkpoint
4. Prepare codec-degraded dataset
5. Pre-encode VAE latents
6. Fine-tune
7. Plot loss curves

In [ ]:
# Cell 1: Install dependencies
!pip install torch==2.4.0 torchaudio==2.4.0 torchvision==0.19.0 -q
!pip install diffusers==0.30.0 transformers==4.44.0 accelerate==0.34.2 -q
!pip install safetensors datasets librosa soundfile tqdm pandas pyyaml -q
!pip install huggingface_hub -q

In [ ]:
# Cell 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DRIVE = '/content/drive/MyDrive/CS614_Project'
os.makedirs(PROJECT_DRIVE, exist_ok=True)
os.makedirs(f'{PROJECT_DRIVE}/checkpoints', exist_ok=True)
os.makedirs(f'{PROJECT_DRIVE}/data', exist_ok=True)
os.makedirs(f'{PROJECT_DRIVE}/outputs', exist_ok=True)
print(f'Project Drive: {PROJECT_DRIVE}')

In [ ]:
# Cell 3: Clone project repo (or upload)
# Option A: Clone from your git repo
# !git clone https://github.com/YOUR_USER/CS614.git /content/CS614

# Option B: Upload Project/ folder manually or copy from Drive
# For now, clone SonicMaster directly
if not os.path.exists('/content/sonicmaster'):
    !git clone https://github.com/AMAAI-Lab/SonicMaster.git /content/sonicmaster

# Copy our custom scripts from Drive if they exist, otherwise we'll create them
import sys
sys.path.insert(0, '/content/sonicmaster')
print('SonicMaster code ready.')

In [ ]:
# Cell 4: Authenticate with HuggingFace (needed for Stable Audio VAE)
from huggingface_hub import login
# Paste your HF token when prompted (needs access to stabilityai/stable-audio-open-1.0)
login()

In [ ]:
# Cell 5: Download pretrained SonicMaster checkpoint
CKPT_PATH = f'{PROJECT_DRIVE}/checkpoints/model.safetensors'

if not os.path.exists(CKPT_PATH):
    from huggingface_hub import hf_hub_download
    print('Downloading SonicMaster checkpoint (~3.6 GB)...')
    hf_hub_download(
        repo_id='amaai-lab/SonicMaster',
        filename='model.safetensors',
        local_dir=f'{PROJECT_DRIVE}/checkpoints',
    )
    print('Done!')
else:
    print(f'Checkpoint already exists: {CKPT_PATH}')

In [ ]:
# Cell 6: Download sample clean audio for dataset preparation
# Using MUSDB18-HQ test set (smaller, ~2.4 GB) or any clean music you upload
#
# Option A: Use your own clean WAVs (upload to Drive under data/clean/)
# Option B: Download a free music subset

CLEAN_DIR = f'{PROJECT_DRIVE}/data/clean'
DATA_DIR = f'{PROJECT_DRIVE}/data'

if not os.path.exists(CLEAN_DIR) or len(os.listdir(CLEAN_DIR)) == 0:
    os.makedirs(CLEAN_DIR, exist_ok=True)
    print('Upload clean WAV/FLAC files to:', CLEAN_DIR)
    print('Or download MUSDB18-HQ test set:')
    print('  wget https://zenodo.org/records/3338373/files/musdb18hq.zip')
    print('  (extract and copy test/ stems to', CLEAN_DIR, ')')
else:
    n = len([f for f in os.listdir(CLEAN_DIR) if f.endswith(('.wav', '.flac', '.mp3'))])
    print(f'Found {n} audio files in {CLEAN_DIR}')

In [ ]:
# Cell 7: Generate codec-degraded dataset
import json, random, os
import numpy as np
import torch
import torchaudio
import soundfile as sf
from pathlib import Path
from tqdm import tqdm

SAMPLE_RATE = 44100
CLIP_DURATION = 30
CLIP_SAMPLES = SAMPLE_RATE * CLIP_DURATION
NUM_CLIPS = 1000  # Adjust based on how much data you have
VAL_RATIO = 0.1

MP3_BITRATES = [24000, 32000, 48000, 64000, 96000, 128000]
OGG_QUALITIES = [-1, 0, 1, 2, 3, 5]

PROMPTS_MP3 = [
    "restore audio compressed at {br}kbps MP3",
    "remove MP3 compression artifacts from {br}kbps audio",
    "enhance low bitrate MP3 audio to high fidelity",
    "fix MP3 compression artifacts and restore high frequencies",
]
PROMPTS_OGG = [
    "restore audio compressed with OGG Vorbis at quality {q}",
    "remove OGG compression artifacts from low quality audio",
    "enhance OGG-compressed audio to high fidelity",
]
ALT_PROMPTS = [
    "make this audio sound better",
    "improve the sound quality",
    "restore the original audio quality",
    "enhance this compressed audio",
]

def find_audio(d):
    exts = {'.wav', '.flac', '.mp3', '.ogg'}
    return sorted([os.path.join(r,f) for r,_,fs in os.walk(d) for f in fs if Path(f).suffix.lower() in exts])

def load_prep(p):
    w, sr = torchaudio.load(p)
    if sr != SAMPLE_RATE: w = torchaudio.functional.resample(w, sr, SAMPLE_RATE)
    if w.shape[0]==1: w = w.repeat(2,1)
    elif w.shape[0]>2: w = w[:2]
    return w

def rand_clip(w):
    T = w.shape[1]
    if T >= CLIP_SAMPLES:
        s = random.randint(0, T-CLIP_SAMPLES)
        return w[:, s:s+CLIP_SAMPLES]
    return torch.nn.functional.pad(w, (0, CLIP_SAMPLES-T))

random.seed(42)
files = find_audio(CLEAN_DIR)
assert files, f'No audio files in {CLEAN_DIR}'
print(f'{len(files)} source files')

clean_out = f'{DATA_DIR}/clean_clips'; os.makedirs(clean_out, exist_ok=True)
deg_out = f'{DATA_DIR}/degraded_clips'; os.makedirs(deg_out, exist_ok=True)

records = []
for i in tqdm(range(NUM_CLIPS)):
    try:
        w = load_prep(random.choice(files))
        clip = rand_clip(w)
        peak = clip.abs().max()
        if peak > 0: clip = clip / peak * 0.95
        
        if random.random() < 0.7:
            br = random.choice(MP3_BITRATES)
            deg = torchaudio.functional.apply_codec(clip, SAMPLE_RATE, format='mp3', compression=br//1000)
            prompt = random.choice(PROMPTS_MP3).format(br=br//1000)
        else:
            q = random.choice(OGG_QUALITIES)
            deg = torchaudio.functional.apply_codec(clip, SAMPLE_RATE, format='ogg', compression=q)
            prompt = random.choice(PROMPTS_OGG).format(q=q)
        
        if deg.shape[1] != clip.shape[1]:
            deg = deg[:, :clip.shape[1]] if deg.shape[1] > clip.shape[1] else torch.nn.functional.pad(deg, (0, clip.shape[1]-deg.shape[1]))
        
        cn = f'clip_{i:05d}_clean.flac'
        dn = f'clip_{i:05d}_degraded.flac'
        sf.write(f'{clean_out}/{cn}', clip.numpy().T, SAMPLE_RATE)
        sf.write(f'{deg_out}/{dn}', deg.numpy().T, SAMPLE_RATE)
        records.append({'prompt': prompt, 'alt_prompt': random.choice(ALT_PROMPTS),
                        'original_location': f'{clean_out}/{cn}', 'location': f'{deg_out}/{dn}', 'duration': 30})
    except Exception as e:
        print(f'Skip {i}: {e}')

random.shuffle(records)
nv = max(1, int(len(records)*VAL_RATIO))
for path, recs in [(f'{DATA_DIR}/trainset.jsonl', records[nv:]), (f'{DATA_DIR}/valset.jsonl', records[:nv])]:
    with open(path,'w') as f:
        for r in recs: f.write(json.dumps(r)+'\n')
print(f'\n{len(records)-nv} train + {nv} val clips generated.')

In [ ]:
# Cell 8: Pre-encode VAE latents
from diffusers import AutoencoderOobleck

device = 'cuda' if torch.cuda.is_available() else 'cpu'
vae = AutoencoderOobleck.from_pretrained('stabilityai/stable-audio-open-1.0', subfolder='vae').to(device)
vae.eval()
_ = vae.requires_grad_(False)

def pad_wav(w, n):
    return w[:n] if w.shape[0]>=n else torch.cat([w, torch.zeros(n-w.shape[0])])

def load_audio_for_vae(path):
    w, sr = torchaudio.load(path)
    if sr != 44100: w = torchaudio.functional.resample(w, sr, 44100)
    if w.shape[0]==1: w = w.repeat(2,1)
    elif w.shape[0]>2: w = w[:2]
    t = 44100*30
    return torch.stack([pad_wav(w[0],t), pad_wav(w[1],t)])

BATCH_SIZE = 4

for jsonl_path in [f'{DATA_DIR}/trainset.jsonl', f'{DATA_DIR}/valset.jsonl']:
    with open(jsonl_path) as f:
        records = [json.loads(l) for l in f if l.strip()]
    
    clean_pt_dir = f'{DATA_DIR}/clean_pt'; os.makedirs(clean_pt_dir, exist_ok=True)
    deg_pt_dir = f'{DATA_DIR}/degraded_pt'; os.makedirs(deg_pt_dir, exist_ok=True)
    
    updated = []
    batch_c, batch_d, batch_r = [], [], []
    
    for rec in tqdm(records, desc=f'Encoding {os.path.basename(jsonl_path)}'):
        try:
            cw = load_audio_for_vae(rec['original_location'])
            dw = load_audio_for_vae(rec['location'])
            batch_c.append(cw); batch_d.append(dw); batch_r.append(rec)
            
            if len(batch_c) >= BATCH_SIZE:
                with torch.no_grad():
                    cl = vae.encode(torch.stack(batch_c).to(device)).latent_dist.mode().transpose(1,2).cpu()
                    dl = vae.encode(torch.stack(batch_d).to(device)).latent_dist.mode().transpose(1,2).cpu()
                for j, r in enumerate(batch_r):
                    cn = os.path.basename(r['original_location']).rsplit('.',1)[0]+'.pt'
                    dn = os.path.basename(r['location']).rsplit('.',1)[0]+'.pt'
                    cp = f'{clean_pt_dir}/{cn}'; dp = f'{deg_pt_dir}/{dn}'
                    torch.save(cl[j], cp); torch.save(dl[j], dp)
                    updated.append({**r, 'original_location': cp, 'location': dp})
                batch_c, batch_d, batch_r = [], [], []
        except Exception as e:
            print(f'  Skip: {e}')
    
    # Flush remaining
    if batch_c:
        with torch.no_grad():
            cl = vae.encode(torch.stack(batch_c).to(device)).latent_dist.mode().transpose(1,2).cpu()
            dl = vae.encode(torch.stack(batch_d).to(device)).latent_dist.mode().transpose(1,2).cpu()
        for j, r in enumerate(batch_r):
            cn = os.path.basename(r['original_location']).rsplit('.',1)[0]+'.pt'
            dn = os.path.basename(r['location']).rsplit('.',1)[0]+'.pt'
            cp = f'{clean_pt_dir}/{cn}'; dp = f'{deg_pt_dir}/{dn}'
            torch.save(cl[j], cp); torch.save(dl[j], dp)
            updated.append({**r, 'original_location': cp, 'location': dp})
    
    pt_jsonl = jsonl_path.replace('.jsonl', '_pt.jsonl')
    with open(pt_jsonl, 'w') as f:
        for r in updated: f.write(json.dumps(r)+'\n')
    print(f'  {len(updated)} encoded -> {pt_jsonl}')

del vae
torch.cuda.empty_cache()
print('\nLatent encoding complete!')

In [ ]:
# Cell 9: Write fine-tuning config pointing to our data
import yaml

finetune_config = {
    'paths': {
        'train_file': f'{DATA_DIR}/trainset_pt.jsonl',
        'val_file': f'{DATA_DIR}/valset_pt.jsonl',
        'test_file': f'{DATA_DIR}/valset_pt.jsonl',
        'infer_file': f'{DATA_DIR}/valset_pt.jsonl',
        'resume_from_checkpoint': '',
        'output_dir': f'{PROJECT_DRIVE}/outputs/finetune_codec',
    },
    'training': {
        'per_device_batch_size': 2,
        'learning_rate': 1e-5,
        'gradient_accumulation_steps': 4,
        'num_train_epochs': 30,
        'num_warmup_steps': 200,
        'max_audio_duration': 30,
    },
    'model': {
        'num_layers': 6,
        'num_single_layers': 18,
        'in_channels': 64,
        'attention_head_dim': 128,
        'joint_attention_dim': 1024,
        'num_attention_heads': 8,
        'audio_seq_len': 645,
        'max_duration': 30,
        'uncondition': False,
        'text_encoder_name': 'google/flan-t5-large',
    },
}

config_path = '/content/finetune_config.yaml'
with open(config_path, 'w') as f:
    yaml.dump(finetune_config, f, default_flow_style=False)
print(f'Config written to {config_path}')

In [ ]:
# Cell 10: Write accelerator config for single GPU
import json

accel_config = {
    'compute_environment': 'LOCAL_MACHINE',
    'distributed_type': 'NO',
    'main_process_port': 29512,
    'downcast_bf16': False,
    'machine_rank': 0,
    'main_training_function': 'main',
    'mixed_precision': 'fp16',
    'num_machines': 1,
    'num_processes': 1,
    'rdzv_backend': 'static',
    'same_network': True,
    'tpu_use_cluster': False,
    'tpu_use_sudo': False,
    'use_cpu': False,
}

accel_path = '/content/accelerator_config.yaml'
with open(accel_path, 'w') as f:
    json.dump(accel_config, f, indent=2)
print(f'Accelerator config written to {accel_path}')

In [ ]:
# Cell 11: Write fine-tuning script to disk
# (This writes the finetune.py script that uses our adapted training loop)
# If you've uploaded the Project folder from your repo, skip this cell.

FINETUNE_SCRIPT = '/content/finetune.py'

# Check if the script was already uploaded
if os.path.exists(FINETUNE_SCRIPT):
    print(f'{FINETUNE_SCRIPT} already exists. Skipping write.')
else:
    print('Please upload finetune.py from your Project/ folder, or run:')
    print('  !cp /content/drive/MyDrive/CS614_Project/finetune.py /content/')

In [ ]:
# Cell 12: Launch fine-tuning!
!accelerate launch \
    --config_file /content/accelerator_config.yaml \
    /content/sonicmaster/train_ptload_inference.py \
    --config /content/finetune_config.yaml \
    --load_from_checkpoint {CKPT_PATH} \
    --learning_rate 1e-5 \
    --lr_scheduler_type cosine \
    --save_every 5 \
    --seed 42

In [ ]:
# Cell 13: Plot training loss curves
import matplotlib.pyplot as plt
import json

output_dir = f'{PROJECT_DRIVE}/outputs/finetune_codec'
summary_file = f'{output_dir}/summary.jsonl'

epochs, train_losses, val_losses = [], [], []
with open(summary_file) as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        try:
            d = json.loads(line)
            if 'epoch' in d and 'epoch/train_loss' in d:
                epochs.append(d['epoch'])
                train_losses.append(d['epoch/train_loss'])
                val_losses.append(d['epoch/val_loss'])
        except:
            pass

fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.plot(epochs, train_losses, label='Train Loss', marker='o', markersize=3)
ax.plot(epochs, val_losses, label='Val Loss', marker='s', markersize=3)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('SonicMaster Fine-tuning: Codec Restoration')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{output_dir}/loss_curves.png', dpi=150)
plt.show()
print(f'Best val loss: {min(val_losses):.4f} at epoch {epochs[val_losses.index(min(val_losses))]}')